# MATH840 — Week 4 practice: the toolbox, end to end

**This session is not marked.** We build one forecast together, on a series everyone has, and argue
about every decision on the way. Then you get your own series and do the same at home — that is
Challenge 1, due 23:59 today.

The recipe below is the whole challenge:

| Block | What we do | Minutes |
|---|---|---|
| 1 | Split the history before touching a model, and find the benchmark | 15 |
| 2 | Try to beat it with the Week 3 toolbox, and see it fail on another series | 20 |
| 3 | Ask the residuals what is left | 15 |
| 4 | Package a submission and check its format | 15 |
| 5 | Issue your own series and look at it | 10 |

Type along. Every block ends with a question — the answers are the point of the session.

## 0. Setup

In [ ]:
!pip install -q statsforecast utilsforecast coreforecast

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from coreforecast.scalers import boxcox, boxcox_lambda

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)

Our shared series: **Australian beer production**, quarterly — 55 years of it. No challenge series
comes from this file, so nothing we do here hands anyone an answer.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")

production = pd.read_csv(f"{BASE}/aus_production_long.csv", parse_dates=["ds"])
beer = (production.query("unique_id == 'Beer'")[["ds", "y"]]
        .dropna().sort_values("ds").reset_index(drop=True))

M, H = 4, 8                      # quarterly data; the challenge horizon for quarterly series
y = beer["y"].to_numpy(float)
dates = beer["ds"]
print(f"{len(beer)} quarters, {dates.min().date()} to {dates.max().date()}")

fig, ax = plt.subplots()
ax.plot(dates, y, color="#314f4f", linewidth=1)
ax.set_title("Australian beer production")
ax.set_ylabel("Megalitres")
plt.show()

## 1. Split first, model second

The rule that makes everything afterwards honest: **the last `H` observations are not data any
more.** They are the exam. We fit on everything before them, and nothing we do may look at them
until we are ready to be scored.

In [ ]:
train, valid = y[:-H], y[-H:]
train_dates, valid_dates = dates.iloc[:-H], dates.iloc[-H:]

print(f"train: {len(train)} quarters up to {train_dates.iloc[-1].date()}")
print(f"valid: {len(valid)} quarters, {valid_dates.iloc[0].date()} to {valid_dates.iloc[-1].date()}")

fig, ax = plt.subplots()
ax.plot(train_dates, train, color="#314f4f", linewidth=1, label="train")
ax.plot(valid_dates, valid, color="#e64173", linewidth=2, label="validation")
ax.legend()
ax.set_title("What we may look at, and what we may not")
plt.show()

### The four methods, and the one number to beat

Same four as Lab 2, same `MASE` scale the scorer uses: the in-sample seasonal naive error of the
whole history.

In [ ]:
def benchmark_forecasts(history, h, m):
    T = len(history)
    return {
        "mean":   np.repeat(history.mean(), h),
        "naive":  np.repeat(history[-1], h),
        "snaive": np.array([history[-m + (i % m)] for i in range(h)]),
        "drift":  history[-1] + np.arange(1, h + 1) * (history[-1] - history[0]) / (T - 1),
    }


SCALE = float(np.mean(np.abs(y[M:] - y[:-M])))          # MASE denominator, fixed for this series


def mase(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))) / SCALE)


def rmse(actual, forecast):
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


simple = benchmark_forecasts(train, H, M)
scores = pd.DataFrame({name: {"rmse": rmse(valid, fc), "mase": mase(valid, fc)}
                       for name, fc in simple.items()}).T
BENCHMARK = scores["rmse"].idxmin()
print(scores.round(3).to_string())
print(f"\nbenchmark: {BENCHMARK}  (MASE {scores.loc[BENCHMARK, 'mase']:.3f})")

In [ ]:
fig, ax = plt.subplots()
ax.plot(dates.iloc[-40:], y[-40:], color="#314f4f", linewidth=1, label="actual")
for name, fc in simple.items():
    style = "-" if name == BENCHMARK else "--"
    ax.plot(valid_dates, fc, style, linewidth=2 if name == BENCHMARK else 1, label=name)
ax.legend(ncol=5)
ax.set_title("The four simple methods over the validation window")
plt.show()

::: Discussion
**Before you look at the table: which of the four should win here, and what in the plot tells you?**

And then: the benchmark's MASE is a number between 0 and something. What would a MASE of exactly 1
mean? Is a MASE above 1 possible, and what would it say about the series?
:::

## 2. Beat it with the toolbox

Three moves from Week 3, in the order worth trying them.

**Move 1: does the variance need stabilising?** If the seasonal swing grows with the level, a
transformation makes the rest of the work easier.

In [ ]:
lam = boxcox_lambda(train, method="loglik", season_length=M)
print(f"Box-Cox lambda on the training data: {lam:.3f}")

early, late = train[:40], train[-40:]
print(f"seasonal swing, first 10 years: {early.std():7.1f}   last 10 years: {late.std():7.1f}")
print(f"level,          first 10 years: {early.mean():7.1f}   last 10 years: {late.mean():7.1f}")

**Move 2: forecast the parts, not the whole.** STL splits the series into trend, season and
remainder. Forecast the seasonally adjusted series with a simple method, carry the season forward
with `snaive`, and add them back.

In [ ]:
def decomposition_forecast(history, h, m, level_method="drift", seasonal=13, robust=True):
    """STL the history, forecast the seasonally adjusted part, put the season back."""
    fit = STL(history, period=m, seasonal=seasonal, robust=robust).fit()
    adjusted = history - fit.seasonal
    level_fc = benchmark_forecasts(adjusted, h, m)[level_method]
    season_fc = np.array([fit.seasonal[-m + (i % m)] for i in range(h)])
    return level_fc + season_fc, fit


candidates = {}
for level_method in ("naive", "drift", "mean"):
    fc, _ = decomposition_forecast(train, H, M, level_method)
    candidates[f"STL + {level_method}"] = fc

candidates[f"{BENCHMARK} (benchmark)"] = simple[BENCHMARK]
candidates[f"STL + drift, averaged with {BENCHMARK}"] = (candidates["STL + drift"] + simple[BENCHMARK]) / 2

table = pd.DataFrame({name: {"rmse": rmse(valid, fc), "mase": mase(valid, fc)}
                      for name, fc in candidates.items()}).T.sort_values("mase")
print(table.round(3).to_string())

In [ ]:
best = table.index[0]
fig, ax = plt.subplots()
ax.plot(dates.iloc[-40:], y[-40:], color="#314f4f", linewidth=1, label="actual")
ax.plot(valid_dates, candidates[best], color="#20B2AA", linewidth=2, label=best)
ax.plot(valid_dates, simple[BENCHMARK], "--", color="#e64173", linewidth=2,
        label=f"benchmark: {BENCHMARK}")
ax.legend()
ax.set_title("Best toolbox candidate against the benchmark")
plt.show()

print(f"skill of '{best}' against the benchmark: "
      f"{mase(valid, candidates[best]) / mase(valid, simple[BENCHMARK]):.3f}")

### The same recipe on a different series

Two lines change — the series, and nothing else. Cement, same file, same quarters, same horizon.

In [ ]:
cement = (production.query("unique_id == 'Cement'")[["ds", "y"]]
          .dropna().sort_values("ds").reset_index(drop=True))
yc = cement["y"].to_numpy(float)
train_c, valid_c = yc[:-H], yc[-H:]
scale_c = float(np.mean(np.abs(yc[M:] - yc[:-M])))
mase_c = lambda fc: float(np.mean(np.abs(valid_c - np.asarray(fc))) / scale_c)

simple_c = benchmark_forecasts(train_c, H, M)
bench_c = min(simple_c, key=lambda k: rmse(valid_c, simple_c[k]))
stl_c, _ = decomposition_forecast(train_c, H, M, "drift")

print(f"cement benchmark: {bench_c}, MASE {mase_c(simple_c[bench_c]):.2f}")
print(f"cement STL + drift:       MASE {mase_c(stl_c):.2f}")
print(f"skill: {mase_c(stl_c) / mase_c(simple_c[bench_c]):.2f}  <- above 1: the toolbox lost")

fig, ax = plt.subplots()
ax.plot(cement["ds"].iloc[-40:], yc[-40:], color="#314f4f", linewidth=1, label="actual")
ax.plot(cement["ds"].iloc[-H:], stl_c, color="#20B2AA", linewidth=2, label="STL + drift")
ax.plot(cement["ds"].iloc[-H:], simple_c[bench_c], "--", color="#e64173", linewidth=2,
        label=f"benchmark: {bench_c}")
ax.legend()
ax.set_title("Cement: the validation window is the 2008 collapse")
plt.show()

::: Discussion
**Why does the decomposition forecast beat a plain seasonal naive on beer and lose on cement?**

The averaged forecast is in the table too. Averaging two forecasts cannot be better than the better
of the two on every observation, yet it often wins on the total. How?

And the one that matters for tonight: **if your series behaves like cement, what goes in your
justification block?** The rubric pays for the reasoning, not for the win.
:::

## 3. Ask the residuals

Residuals are what the model could not explain. We want them to look like noise: no autocorrelation,
zero mean, stable variance. Anything else is structure we left on the table.

In [ ]:
fit_full = STL(train, period=M, seasonal=13, robust=True).fit()
adjusted = train - fit_full.seasonal

# One-step residuals of the level model we chose: drift on the adjusted series.
one_step = np.array([benchmark_forecasts(adjusted[:t], 1, M)["drift"][0]
                     for t in range(M + 1, len(adjusted))])
resid = adjusted[M + 1:] - one_step

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].plot(train_dates.iloc[M + 1:], resid, color="#FFA500", linewidth=0.9)
axes[0].set_title("Residuals over time")
plot_acf(resid, lags=24, ax=axes[1], title="ACF of residuals")
axes[2].hist(resid, bins=25, color="#20B2AA")
axes[2].set_title("Distribution")
plt.tight_layout()
plt.show()

lb = acorr_ljungbox(resid, lags=[2 * M], model_df=0)
print(lb.round(4).to_string())
print(f"residual mean {resid.mean():.2f}, sd {resid.std(ddof=1):.2f}")

::: Discussion
**The Ljung-Box p-value is what it is — say out loud what it licenses us to conclude, and what it
does not.**

Two specific questions. If the test rejects, is the forecast we just built useless? And which part
of our submission is most damaged by autocorrelated residuals — the point forecast, or the interval?
:::

## 4. Package a submission

Now, and only now, the validation window goes back into the data. We refit on the whole history and
forecast the `H` steps nobody has seen.

In [ ]:
FINAL, _ = decomposition_forecast(y, H, M, level_method="drift")

# A crude, honest interval: the spread of the one-step residuals, widening with the horizon.
sigma = resid.std(ddof=1)
steps = np.arange(1, H + 1)
lo, hi = FINAL - 1.28 * sigma * np.sqrt(steps), FINAL + 1.28 * sigma * np.sqrt(steps)

FUTURE = pd.date_range(dates.iloc[-1], periods=H + 1, freq=pd.infer_freq(dates))[1:]

fig, ax = plt.subplots()
ax.plot(dates.iloc[-40:], y[-40:], color="#314f4f", linewidth=1, label="history")
ax.plot(FUTURE, FINAL, color="#20B2AA", linewidth=2, label="forecast")
ax.fill_between(FUTURE, lo, hi, color="#20B2AA", alpha=0.2, label="80% interval")
ax.legend()
ax.set_title("What we would submit")
plt.show()

In [ ]:
submission = pd.DataFrame({
    "student_id": "DEMO",
    "code": "DEMO-BEER",
    "ds": FUTURE.strftime("%Y-%m-%d"),
    "yhat": FINAL,
    "yhat_lo_80": lo,
    "yhat_hi_80": hi,
})
submission.to_csv("DEMO_ch1_forecast.csv", index=False)

check = pd.read_csv("DEMO_ch1_forecast.csv", parse_dates=["ds"])
assert len(check) == H
assert list(check.columns) == ["student_id", "code", "ds", "yhat", "yhat_lo_80", "yhat_hi_80"]
assert check[["yhat", "yhat_lo_80", "yhat_hi_80"]].notna().all().all()
print(submission.to_string(index=False))
print("\nformat OK")

::: Discussion
**We just refitted on data that includes the validation window. Why is that not cheating — and what
number from Section 2 have we now lost the right to quote as an expectation?**
:::

## 5. Your own series

Run this. It turns the identifier you have used since Week 1 into your series for the rest of the
course, and downloads it. Say one sentence out loud about what you see: that sentence is the first
line of your justification block.

In [ ]:
MY_ID = ""   # <-- your identifier from Week 1

TRACKB = f"{BASE}/trackb"


def assign_series(student_id, codes):
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return codes[int(digest, 16) % len(codes)]


if MY_ID.strip():
    index = pd.read_csv(f"{TRACKB}/index.csv")
    CODE = assign_series(MY_ID, sorted(index["code"]))
    spec = index.set_index("code").loc[CODE]
    mine = pd.read_csv(f"{TRACKB}/{CODE}.csv", parse_dates=["ds"])
    print(f"your series: {CODE} | m = {spec['m']} | h = {spec['h']} | {len(mine)} observations "
          f"| {mine['ds'].min().date()} to {mine['ds'].max().date()}")

    fig, ax = plt.subplots()
    ax.plot(mine["ds"], mine["y"], color="#6A5ACD", linewidth=1)
    ax.set_title(f"Series {CODE}")
    plt.show()
else:
    print("set MY_ID to the identifier you used in Week 1")

### What to do at home

The challenge is this notebook, on your series, with an argument attached:

1. the plots, and what they show — **Section 1**;
2. the split, the benchmark, your candidates, the winner — **Section 2**;
3. why, residual diagnostics, what you expect — **Section 3**;
4. the submission file, written by the notebook — **due 23:59 today**.

The template has the same helpers as this notebook:
[Challenge 1 template](https://colab.research.google.com/github/Aranaur/aranaur.rbind.io/blob/main/lectures/kse/MATH840/26autumn/labs/_lab04.ipynb).